## Graphs!  

actually similar to trees

In [1]:
from typing import List, Optional

<b> 127. Word Ladder </b>

only change from word to words that have exactly one letter difference. in this way the problem can be transformed into a graph problem, begin word => visit its neighbors (defined as word with only one char difference) => non-visited neighbors' neighbors => until we arrived at the end word.

so it is a breath first search structure. (BFS)

because of the graph defines the path to visit, and the "step" var keeps track of how many steps taken, the first time when we see the "endword", it is actually the shorted path, so no need to visit all the possible paths inorder to know the shorted one.

- time complexity:
    -   construct graph costs O(N*L) N is the length of wordList, L is the avg length of word in the list
    -   the BFS in worse case will visit all items in the graph, so O(N*L)
 
- space complexity
    -   the graph: O(NL)
    -   the queue and visited set at most stores N word, so O(N)
    -   in total O(NL)

In [1]:
from typing import List, Optional

In [3]:
from collections import deque
class Solution:
    def ladderLength(self, beginWord, endWord, wordList):
        def construct_neighbors(word_list):
            d = {}
            for word in word_list:
                for i in range(len(word)):
                    # no need to worry i+1 is out of bound, cz slicing wont give index out of bound error
                    key = word[:i]+'_'+word[i+1:]
                    d[key] = d.get(key, [])+[word]  # {'_ot': ['lot', 'dot', 'hot']}
            return d
        
        def bfs_words(begin, end, word_dict):
            # queue has word and level => step
            # why initiate the step as 1? => example "hit" -> "hot" -> "dot" -> "dog" -> "cog", result should be 5, not 4
            queue, visited = deque([(begin, 1)]), set()
            while queue:
                word, step = queue.popleft()  # this is why using deque
                if word not in visited:  # this check makes the execution much much faster
                    visited.add(word)
                    if word == end:
                        return step
                    # otherwise find its neighbors and add to queue
                    for i in range(len(word)):
                        s = word[:i] + '_' + word[i+1:]
                        neighbors = word_dict.get(s, [])
                        for n_word in neighbors:
                            if n_word not in visited:
                                queue.append((n_word, step+1))
            return 0
        
        if endWord not in wordList:
            return 0
        word_dict = construct_neighbors(set(wordList))
        return bfs_words(beginWord, endWord, word_dict)
            
            
            
            
            

In [4]:
beginWord = "hit"
endWord = "cog"
word_dict = ["hot","dot","dog","lot","log","cog"]
a= Solution()
a.ladderLength(beginWord, endWord, word_dict)

5

bidirectional BFS. Search both from start word and end word, if they meet at some intermediate word, that means existing such a road from start word to end word

In [9]:
from collections import deque, defaultdict
class Solution:
    def ladderLength(self, beginWord, endWord, wordList):
        """bidirectional BFS"""
        
        def find_words(queue, one_visited, two_visited):
            """
            queue: [(word, level)], deque
            one_visited: {word: level} current direction
            two_visited; {word: level} the other direction
            """
            word, step = queue.popleft()
            for i in range(L):
                intermediate = word[:i]+'_'+word[i+1:]
                
                for neighbor in neighbors_dict[intermediate]:
                    if neighbor in two_visited:
                        return step + two_visited[neighbor]
                    if neighbor not in one_visited:
                        one_visited[neighbor] = step + 1
                        queue.append((neighbor, step+1))
            return
        
        if endWord not in wordList:
            return 0
        
        L = len(beginWord)
        
        # construct neighbors dictionary
        neighbors_dict = defaultdict(list)
        for word in wordList:
            for i in range(L):  # only need to loop L time
                key = word[:i]+'_'+word[i+1:]
                neighbors_dict[key].append(word)
        
        queue_begin = deque([(beginWord, 1)])
        queue_end = deque([(endWord, 1)])
        visited_begin = {beginWord: 1}
        visited_end = {endWord: 1}
        # if one queue is empty the other is not, but they still dont meet, that means no such a way
        while queue_begin and queue_end:
            # queue_begin, queue_end, visited_begin, visited_end are all reference and are updated in side of the
            # execution of find_words() method
            # go forward one level from begin word
            res = find_words(queue_begin, visited_begin, visited_end)
            if res:
                return res
            # go forward one level from end word
            res = find_words(queue_end, visited_end, visited_begin)
            if res:
                return res
        return 0

In [10]:
beginWord = "hit"
endWord = "cog"
wordList = ["hot","dot","dog","lot","log","cog"]
b= Solution()
b.ladderLength(beginWord, endWord, wordList)

5

<b> 126 Word Ladder II </b>

instead of returning total step count, return all possible path.  
if simply modifying the above 125 solution by appending (word, [word route]) even with next_queue to avoid visiting the next unnecessary level, it wont produce all the valid paths, because the endword is added to the queue once. 

here using the BFS search, and the layer dictionary store: 
- {word: [all_path_ending_with_this_word]}
-  if not using a dictionary to store the list of paths ending at word, when we do the visited.add(ending_word), it wont process the next one in the queue (if it is a list, like (c, (path1), (c, path2)), because c is in the visited set, the path2 wont be counted, hence a dictionary is better)

- pay attention to when the word is added to the visited set (this does not matter)
    -   it is added when looping over the layer, instead of adding it when checking the neighbor. This is important because in this way the next layer could potentially have record that otherwise wont exist if adding it in neighbor call
    -   for example, check the layer records below
    

Still a Breath Frist Search solution, but this time, instead of using a simple queue (node, step), we create a dictionary, key as the last visited word, value is a list of path that ends with this last word. So when we hit the "end-word", we actually have access to all path that ends with this end_word.

one difference from previous problem 127 is to reconstruct the path dictionary for every level

input: begin: "hit", end: "cog"

starting layer: {"hit": [["hit"]]}

layer 2:  
defaultdict(<class 'list'>, {'hot': [['hit', 'hot']]})  
layer 3:  
defaultdict(<class 'list'>, {'dot': [['hit', 'hot', 'dot']], 'lot': [['hit', 'hot', 'lot']]})  

layer 4:  see the lot appears in both layer 3 and 4, that is due to the visited set is updated only when looping over the layer, instead of during next layer construction (neighbor checking), otherwise the key in current layer wont be able to appear in next layer

defaultdict(<class 'list'>, {'lot': [['hit', 'hot', 'dot', 'lot']], 'dog': [['hit', 'hot', 'dot', 'dog']], 'log': [['hit', 'hot', 'lot', 'log']]})  

layer 5:
defaultdict(<class 'list'>, {'log': [['hit', 'hot', 'dot', 'dog', 'log']], 'cog': [['hit', 'hot', 'dot', 'dog', 'cog'], ['hit', 'hot', 'lot', 'log', 'cog']]})  


however the below implemention hits the memory limit, since the layer dictionary keeps track of all the path.

time complexity
-   N: number of words, L: len of each word, P: number of shortest path
-   construct graph: O(N*L)
-   BFS: visited the node only once, worse case visited all path of the graph, O(NL)
-   repeatedly copy the path for each layer, for worse case there is P path, each path contains N word, each word has size L: O(PNL) -> this is the step that leads to high memory usage and should be optimized
-   so in total O(NL + PNL)

space complexity
-   graph: O(NL)
-   storing all path O(PNL). repeatedly copy the path for each layer, for worse case there is P path, each path contains N word, each word has size L: O(PNL) -> this is the step that leads to high memory usage and should be optimized

In [ ]:
from collections import defaultdict, deque
from string import ascii_lowercase
class Solution:
    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:
        def construct_graph(wordList, beginWord):
            l = len(beginWord)
            d = {}
            for word in wordList:
                for i in range(l):
                    key = word[:i]+'_'+word[i+1:]
                    d[key] = d.get(key, [])+[word]
            return d
                
        if endWord not in wordList:
            return []
        result = []
        word_graph = construct_graph(wordList, beginWord)
        wordList = set(wordList)  # set searching is O(1)
        layer = {}
        layer[beginWord] = [[beginWord]]  # a list of possible path, each list inside ending with the word as key
        visited = set()
        while layer:
            next_layer = defaultdict(list)
            for word in layer:
                if word not in visited:
                    visited.add(word)
                    if word == endWord :
                        return layer[word]
                        # result.extend(path for path in layer[word])
                    else:  
                        # start searching the neighbors
                        for i in range(len(word)):
                            key = word[:i]+'_'+word[i+1:]
                            neighbors = word_graph.get(key, [])
                            for w in neighbors:
                                if w not in visited:  # valid, not visited word
                                    next_layer[w]+=[path+[w] for path in layer[word]]

            layer = next_layer

        return result

to optimize the space complexity, we should eliminate the step of copying all path to each of the layer dictionary, but using a dictionary to store the {"last_word": [previous_word list]} dictionary, then using DFS to reconstruct the path when we hit the end_word

In [ ]:
from collections import defaultdict, deque
from string import ascii_lowercase
class Solution:
    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:
        wordList = set(wordList)  # set searching is O(1
        if endWord not in wordList:
            return []
        
        
        l = len(beginWord)
        word_graph = defaultdict(list)
        for word in wordList|{beginWord}:
            for i in range(l):
                key = word[:i]+'_'+word[i+1:]
                word_graph[key].append(word)
        
        path_dict = defaultdict(list) # {last_word: [previous step word list]}
        global_visited = set() # all historical visited nodes
        found = False
        level = deque([beginWord])

        while level and not found: # there the level will be updated in the loop using next level
            next_level = deque()
            level_visited = set()
            while level:
                word = level.popleft()
                if word == endWord:
                    found = True # finish the whole level where the endWord is found
                for i in range(len(word)):
                    key = word[:i]+'_'+word[i+1:]
                    for nei in word_graph[key]: # this nei word can appear multiple times from different word (the same level)'s neighbor links
                        if nei not in global_visited:
                            if nei not in level_visited:
                                level_visited.add(nei)
                                next_level.append(nei)
                            path_dict[nei].append(word) # this ensures different path ends at nei will be recorded
            level = next_level
            global_visited |= level_visited # this ensures different path ends at same words will be recorded. this is to update the visited nodes at one time at level level

        # reconstruct the path using DFS, from endWord to beginWord
        # path_dict and result are all reference in the dfs function
        result = []
        def dfs(word, path):
            if word == beginWord:
                result.append(path[::-1])
                return
            for prev in path_dict[word]:
                dfs(prev, path+[prev])
        if found:
            dfs(endWord, [endWord])
        return result

<b> 133 clone Graph </b>

using a dictionary to have the old node to new node mapping, while travel the old graph, create the corresponding copied node, then copy the relationship too. the dictionary also serve as the "visited" set

In [2]:
"""
# Definition for a Node.
"""
class Node:
    def __init__(self, val = 0, neighbors = None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

from collections import defaultdict
class Solution:
    def cloneGraph(self, node: 'Node') -> 'Node':
        if not node:
            return None

        new_node_dict = {node: Node(node.val)}
        stack = [node]
        while stack:
            n = stack.pop()
            for neighbor in n.neighbors:
                # smartly using new_node_dict's key to check if visited
                if neighbor not in new_node_dict:
                    stack.append(neighbor)
                    # copy the value
                    new_node_dict[neighbor] = Node(neighbor.val)
                # copy the relationship, new edge could happen for visited nodes
                new_node_dict[n].neighbors.append(new_node_dict[neighbor])
        return new_node_dict[node]

<b> 721 Accounts Merge </b>

a very good graph problem. the graph is {email: [account_index list]}. first loop over the account list, for each email, find other account index from the graph, loop until no more account index should be checked for this account_index.

time complexity :
-   N: total number of accounts, M: total number of email
-   graph construction O(M) visit each email exactly once
-   dfs helper, each account is visited once, for visited account, all emails are looped once, so O(N+M), not that because using set to store the emails, it is okay that same email account are added to the set multiple times. 
-   sort in merging section takes O(KlogK) K is the avg number of emails in merged group, for worse case it is O(MlogM) for case that all emails are connected to a single account
-   so in total O(N+M+MlogM), since MlogM is usually larger, so it is O(MlogM)
space complexity:
-   graph O(M), each email has a list record
-   visited set O(N) for all account
-   res list: O(M) all emails
-   call stack O(N) for worse case that all are connected.
-   in total O(N+M)


In [ ]:
from collections import defaultdict
class Solution:
    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        """DFS"""
        # construct a graph: {email: [account index]} => better than {email: set(neighbor_emails)} also need to keep {email: name} map. but using the account index is more concise
        # dfs, have a visited set to avoid endless loop
        
        graph = defaultdict(list)
        visited = set()
        res = []
        
        # construct graph: {email: [account index]}
        for i, act in enumerate(accounts):
            for email in act[1:]:
                graph[email].append(i)
                
        def dfs_helper(i, emails):
            # a recursive call to merge accounts
            # i: index of the accounts list
            # emails: a set of emails
            if i in visited:
                return
            visited.add(i)
            for e in accounts[i][1:]:
                emails.add(e)
                for act_index in graph[e]:
                    dfs_helper(act_index, emails)
        
        
        for i in range(len(accounts)):
            if i in visited: continue
            name, emails = accounts[i][0], set()
            dfs_helper(i, emails)
            res.append([name]+sorted(emails))
        return res

also can be solved by union find

<b> 207 Course Schedule </b>

the problem is also known as topological sort problem, which is to find a global order for all nodes in a DAG (Directed Acyclic Graph) with regarding to their dependencies.

<b>Topological sort </b>  
construct a graph:{prerequisite: [dependings]}  
construct a degree dictionary: {course: degree integer}: n degree means this course has n prerequisits need to take  
the course with degree 0 is the one able to take first. if no course has 0 degree, the courses are in a loop, cant be finished.
first starting with courses with degree 0, find its depending courses and reduce their degrees by 1. that means number of requisites needs to be finished before able to take the depending course is reduced by 1. If now a new course has degree 0, add it to the start course queue.  
at the end if all courses are taken, each of the degree is 0, sum is 0 too

time complexity
-   N: number of course, node, M: number of prerequisite pairs, edge
-   construct graph takes O(M)
-   start_course at most has N node, internal for loop is related to the number of pairs, the edge, so it is O(N+M) total work is portional to the number of edges and nodes
-   in total O(N+M)

space complexity:
-   graph O(M), degree array O(N)
-   queue: O(N),
-   in total O(N+M)

In [11]:
from collections import defaultdict, deque
def canFinish(numCourses, prerequisites):
    # construct a graph:{prerequisite: [dependings]}
    graph = defaultdict(list)
    # construct a degree dictionary: {course: degree integer}
    # n degree means this course has n prerequisits need to take
    degree = [0]*numCourses
    for d, pre in prerequisites:
        graph[pre].append(d)
        degree[d]+=1 # course number happens to be used as index

    # the course with degree 0 is the able to take first. if no course has 0 degree, the courses are in a loop, cant be finished
    start_courses = deque([i for i in range(numCourses) if degree[i]==0])
    while start_courses:
        pre = start_courses.popleft()
        for j in graph[pre]:
            # j is the depending courses of i
            degree[j]-=1  # number of requisites needs to be finished before able to take the course j is reduced by 1
            if degree[j]==0:
                # able to take it as next course
                start_courses.append(j)
    # if all courses are taken, they will all be 0
    return not sum(degree)

In [12]:
numCourses = 2
prerequisites = [[1,0],[0,1]]
canFinish(numCourses, prerequisites)

False

<b> 210 course schedule II </b>.   
similar as above

In [14]:
from collections import defaultdict, deque
def findOrder(numCourses, prerequisites): 
    graph = defaultdict(list)  # {prerequisite: [depending courses]}
    degree=[0]*numCourses  # [integer] how many prerequisites needs to take

    for course, pre in prerequisites:
        graph[pre].append(course)
        degree[course]+=1 

    result=[]
    # start with course with no prerequisite
    starting_course = deque([i for i in range(numCourses) if degree[i]==0])
    while starting_course:
        course = starting_course.popleft()
        result.append(course)
        # its dependencies' degree - 1
        for d in graph[course]:
            degree[d]-=1
            if degree[d]==0:
                # able to be taken next
                starting_course.append(d)

    if sum(degree)==0:
        # all courses are taken
        return result
    else:
        return []

In [15]:
numCourses = 4
prerequisites=[[1,0],[2,0],[3,1],[3,2]]
findOrder(numCourses, prerequisites)

[0, 1, 2, 3]

<b> 630 course Schedule III </b>

looks like interval problems.

time complexity:
-   N: num of course, also the len of courses list
-   sorting: O(NlogN)
-   for loop: O(N), inside for loop, each heappush operation takes O(logN), every heappop takes O(LogN). each course is pushed and popped at most once, so total is O(NlogN)

space:
-   heap queue: O(N)


In [5]:
import heapq
class Solution:
    def scheduleCourse(self, courses: List[List[int]]) -> int:
        courses = sorted(courses, key =lambda x: x[1])
        pq = []
        start = 0
        for t, end in courses:
            start += t
            # course with longer duration will be on top of queue
            heapq.heappush(pq, -t)
            # when the course added not fit the criteria
            # greedily remove the largest-length cours, thanks to the heapq to keep check what the longest course is
            # by removing the course taking the longest time, can make sure the length of courses taken is the maximal
            while start > end:
                # because the time now because a negative value
                # poping out meanings not taking that course
                start += heapq.heappop(pq)
        return len(pq)

<b> 1136 parallel courses </b>

topological sorting algorithms  
suppose n -> v means n is the prerequisite of v

1. initialize a queue with all ndoes with no incoming edges (no prerequisites)
2. while the queue is not empty
- remove a node from the queue
- add the node to the topological sorting result list
- for each v node with an edge of node u
    - remove the dependency of n to v since n is now visited
    - if v does not have other incoming edges (no other prerequisites), add v to the queue

In [3]:
# relations: List[List[int]]
def minimumSemesters(n: int, relations) -> int:
    """
    transform into a graph, then check the level of the graph during traverasl
    """
    pre_to_next_graph = defaultdict(list)
    preq_count = [0] * (n+1) # index 0 is not used, because label starts from 1

    # construct the graph
    for preq, nextC in relations:
        pre_to_next_graph[preq].append(nextC)
        preq_count[nextC] += 1
    res = 0
    q = [i+1 for i in range(n) if preq_count[i+1]==0]
    while q:
        next_q = []
        for c in q:
            for next_c in pre_to_next_graph[c]:
                preq_count[next_c]-=1
                if preq_count[next_c] == 0:
                    next_q.append(next_c)
        q = next_q
        res += 1
    # return res only for valid case, otherwise -1
    return res if sum(preq_count) == 0 else -1
    
    

In [ ]:

def minimumSemesters(n: int, relations: List[List[int]]) -> int:
    """
    transform into a graph, then check the level of the graph during traverasl
    """
    pre_to_next_graph = defaultdict(list)
    preq_count = [0] * (n+1) # index 0 is not used

    # construct the graph
    for preq, nextC in relations:
        pre_to_next_graph[preq].append(nextC)
        preq_count[nextC] += 1
    
    queue = deque([(1, c) for c, val in enumerate(preq_count) if val == 0 and c!=0]) # (level, course)
    max_semester = 0
    while True:
        next_semester = deque()
        while queue:
            level, course = queue.popleft()
            for next_course in pre_to_next_graph[course]:
                preq_count[next_course] -= 1
                if preq_count[next_course] == 0:
                    next_semester.append((level+1, next_course))
        max_semester += 1
        if not next_semester: break
        queue = next_semester

    if sum(preq_count) > 0: return -1
    return max_semester



In [3]:
from collections import defaultdict, deque

def minimumSemesters(self, n, relations):
    """
    :type n: int
    :type relations: List[List[int]]
    :rtype: int
    """
    # prev to next
    p_to_n = defaultdict(set)
    # next to prev
    n_to_p = defaultdict(set)
    for p, n in relations:
        p_to_n[p].add(n)
        n_to_p[n].add(p)

    queue = deque([]) # will store (level, node) tuple
    for i in range(1, n+1):
        # no prerequisite
        if not n_to_p[i]:
            # for all nodes with no prerequisites, they are from level 1
            queue.append((1, i))
    max_level = 0
    topological_sort = []
    while queue:
        # pop from left so start from low level node
        level, node = queue.popleft()
        # add node to topological ordering
        topological_sort.append(node)
        max_level = max(max_level, level)
        # for each v with and edge u -> v:
            # remove edge u -> v
            # if v has no imcomping edge, add v to the queue
        for next_node in p_to_n[node]:
            # when node is pop and visited, can consider the prerequisite of node is satisfied
            # now check the next node with prerequisite of node, remove the dependency first since it is satisfied
            # then see if this next node has other prerequiste or not, if not, increase the level and put it to queue
            n_to_p[next_node].remove(node)
            if not n_to_p[next_node]:
                queue.append((level+1, next_node))
    # topological_sort records all the node visted, if it visited each of every node, then there is no cycle of those nodes
    return max_level if len(topological_sort) == n else -1


<b> 1494 parallel courses II </b>

The solution is the graph topological sort + recursive with cache. Because for the available M courses with indegree=0, we cant just simply choose the first K courses or K courses with least/most in degree, the order of choosing the K courses for each semester will impact the total semester number needed.

Hence the solution is using a recursive function to try every possible combination for every semester. the lru_cache decoration is used for caching the seen cases.

this solution does not need to use an additional mask var because the in_degree set taken courses to a negative value

In [ ]:
# solution without using mask

from itertools import combinations
from functools import lru_cache
import math

class Solution:
    def __init__(self):
        self.graph = defaultdict(list)
    
    def minNumberOfSemesters(self, n: int, relations: List[List[int]], k: int) -> int:
        in_degree = [0]*n
        for pre, next_c in relations:
            self.graph[pre-1].append(next_c-1)
            in_degree[next_c-1] += 1
        return self.recursive(n, k, tuple(in_degree))  
     
    @lru_cache(None)
    def recursive(self, n, k, in_degree:tuple):
        # base case:
        courses = [ind for ind in range(n) if in_degree[ind] == 0]
        if not courses: return 0

        res = math.inf
        for c in combinations(courses, min(k, len(courses))):
            new_in_degree = list(in_degree)
            for course in c:
                # here the coming course becomes negative, so wont be considered again in child recursion step
                # it acts as the "visited" set
                new_in_degree[course] = -1
                for next_c in self.graph[course]:
                    new_in_degree[next_c] -= 1
            res = min(res, 1 + self.recursive(n, k, tuple(new_in_degree)))
        return res


In [8]:
test = Solution()
test.minNumberOfSemesters(12, [[1,2],[1,3],[7,5],[7,6],[4,8],[8,9],[9,10],[10,11],[11,12]], 2)

6

using bit mask

- initialize the maske to be (1 << n) - 1
    -   1 << n: shift 1 left by n bits, so a signel 1 followed by n zeros, ex: 1 << 4: 0b10000
    -  (1 << n)-1: turns all zero into ones, and the first 1 becomes 0 and drop off. ex 0b10000 - 1 = 0b1111
- checking if i-th class is still available (not taken) using mask: mask & (1 << i)
    -   for example mask: 0b1101, 1 << 4 = 0b10000, so it is 0b01101 & 0b01000 = 0b01000 (AND: both position is 1 then 1)
    -   1<< i, the i-th position is 1, all other position is 0, if the returned value is not 1, then it means in mask, the i-th position is 1 too, so not taken. all others in result is guaranteed to be 0 because the i << n other position except the i-th is all 0.
-   marking a course as taken by fligging the bit using: new_mask = mask ^ (1 << i)
    -   ^ is XOR operation, the result is 1 if the bits are different, 0 if they are the same
-   base case: if not mask: return 0, means all bits are 0, so all courses are taken.

In [9]:
from itertools import combinations
import math
class Solution:
    def __init__(self):
        self.graph = defaultdict(list)

    @lru_cache(None)
    def recurse(self, mask, n, k, in_degree: tuple):
        # base case: all courses are taken
        if not mask: return 0

        # i-th course is not taken and its pre courses are all taken
        courses = [ind for ind in range(n) if in_degree[ind]==0 and mask & (1<<ind)]

        res = math.inf
        for c in combinations(courses, min(k, len(courses))):
            new_mask, new_in_degree = mask, list(in_degree)
            for i in c:
                # flip the i course as taken
                new_mask = new_mask ^ (1 << i)
                for next_c in self.graph[i]:
                    new_in_degree[next_c] -= 1
            res = min(res, 1 + self.recurse(new_mask, n, k, tuple(new_in_degree)))
        return res
    def minNumberOfSemesters(self, n: int, relations: List[List[int]], k: int) -> int:
        in_degree = [0]*n
        for pre, next_c in relations:
            in_degree[next_c - 1] += 1
            self.graph[pre-1].append(next_c-1)
        # using tuple because if passing list it returns unhashable error
        return self.recurse((1<<n)-1, n, k, tuple(in_degree))

<b> 2050 Parallel courses III </b>

adding a time var to keep track of time needed to finish a course. Courses can be taken in parallel so the time needed is the longest one taken among the n courses taken at the same time.

same as previous courses problems, using a graph to keep track of pre_course to next course, and a degree dict / list to track how many pre-courses left. for course with 0 degree then it can be pushed to the queue for consumption. When taking the course, update the max_time dictionary for the longest time needed to finishi this course

time complexity:
-   graph construction: O(E) E is the number of relations
-   first populate the q: O(N), N is the node number
-   while loop for the q: each course is taken at most once, so O(N)
-   max of max_time takes O(N)
-   so total taking O(E+N)

space complexity:
-   graph: O(E+N)
-   q: O(N)
-   max_time: O(N)
-   degree: O(N)
-   total O(E+N)

In [2]:
from collections import defaultdict, deque
from typing import List
class Solution:
    def minimumTime(self, n: int, relations: List[List[int]], time: List[int]) -> int:
        """
        pre_to_next: {n: [1,2,3]} n is 1,2,3's pre course
        pre_degree: {n: m} course n has m different pre needs to be taken first
        time: if 1 takes 3 month, 3 takes 1 month, 4 also 1, then 1,3,4 can just take 3 month in total
        
        when pre_count[n] == 0 -> push n to a queue for next available course to be taken
        using a max_time dict to track the max time reach/ finish course n
        when n is taken, update max_time[n] to be time[n]+max(prefreq time)
        """

        pre_to_next = defaultdict(list)
        pre_degree = [0]*(n+1) # note that course is n, index is n-1
        for x, y in relations:
            pre_to_next[x].append(y)
            pre_degree[y] += 1
        q = deque()
        max_time = [0]*(n+1)
        for node in range(1, n+1):
            if pre_degree[node] == 0:
                q.append(node)
                max_time[node] = time[node-1] # index diff
        
        while q:
            node = q.popleft()
            for next_c in pre_to_next[node]:
                max_time[next_c] = max(max_time[next_c], max_time[node]+time[next_c-1])
                pre_degree[next_c] -= 1
                if pre_degree[next_c] == 0:
                    q.append(next_c)
        # here cannot gurantee n is always the last course
        # max_time is the accumulative time, so the max value is the final time needed
        return max(max_time)






<b> 2101 detonate the maximum bombs </b>

more a graph question, and actually not solved using union find patter.  
first construct a graph with key bomb point to a list of bomb that it can detonate.  
then loop over each node to keep track of how many node it can detonate by going over the graph in dfs manner.  then compare the the count to get max number

time complexity:
- O(N^2)
-   construct graph takes O(N*N), dfs at most visited all nodes so O(N), but the for loop takes O(N) so together O(N*N)


In [5]:
class Solution(object):
    def maximumDetonation(self, bombs):
        """
        :type bombs: List[List[int]]
        :rtype: int
        """

        self.graph = defaultdict(list)
        n = len(bombs)
        # construct a graph of bomb: [list of bomb in its range]
        for i in range(n):
            for j in range(n):
                if i == j: continue
                # j is in i's bomb range
                if (bombs[i][0]-bombs[j][0])**2 + (bombs[i][1]-bombs[j][1])**2 <= bombs[i][2]**2:
                    self.graph[i].append(j)

        def dfs(bomb_ind, visited):
            # node: the bomb
            # a set of bomb that can be triggered by this node bomb
            # finally altered the visited set
            for related_bomb_ind in self.graph[bomb_ind]:
                if related_bomb_ind not in visited:
                    visited.add(related_bomb_ind)
                    dfs(related_bomb_ind, visited)

        res = 0
        for i in range(n):
            # initiated a set for each child, must contain itself as the first one to explode
            visited = set([i])
            dfs(i, visited)
            res = max(res, len(visited))
        return res

            

In [6]:
bombs=[[1,2,3],[2,3,1],[3,4,2],[4,5,3],[5,6,4]]
s = Solution()
s.maximumDetonation(bombs)

5

<b> 1391. Check if There is a Valid Path in a Grid </b>

the grid is a new one, how to define the direction is the key step.

some tricks at how to determine the next step from point (i, j) that lead it ot point (j, k) is a valid step
- get the output direction from point(i, j), 
- check if this output direction can be the input direction of the point (j, k)

In [8]:
from collections import deque
class Solution(object):
    def hasValidPath(self, grid):
        """
        :type grid: List[List[int]]
        :rtype: bool
        """
        # given a grid with size [m,n], the initial location is [0,0], end location is [m-1, n-1]
        # define index change for 4 directions, think of it in a matrix index way, not the traditioal math x y way
        # moving up: row number decrease, col num unchanged
        UP, DOWN, LEFT, RIGHT = (-1, 0), (1, 0), (0, -1), (0, 1)
        m, n = len(grid), len(grid[0]) # get the size of grid
        # key is the grid type, value is a dictionary where its key is possible entry direction, value is corresponding exit direction
        # note that for a grid type there can be two ways of entering it, once the enter direction is known, the exit direction is determined
        type_to_direction = {
            # for type 1, if enter from left direction, will exit from left direction, think of it as an arrow enter left exit left
            # if enter from right direction, will exit from right direction too
            # not the direction of final pointint to where
            # but the direction of coming from where
            # the values of the map is the possible direction for each type of grid
            1: {LEFT: LEFT, RIGHT: RIGHT}, 
            2: {UP: UP, DOWN: DOWN},
            # based on the type 3 grid, left in down out, up in right out
            3: {RIGHT: DOWN, UP: LEFT}, 
            4: {LEFT: DOWN, UP: RIGHT},
            5: {RIGHT: UP, DOWN: LEFT},
            6: {LEFT: UP, DOWN: RIGHT}
        }
        q = deque([(0, 0)])# starting point the left upmost corner
        visited = set([(0, 0)])
        while q:
            # similar to the breadth first search in tree question
            x, y = q.popleft()
            if x == m-1 and y == n-1:
                # reach the target point
                return True
            for dx, dy in type_to_direction[grid[x][y]].values():
                nx, ny = x + dx, y + dy
                # check if new x, y is valid
                # 1) if nx, ny in range
                # 2) if nx, ny not visited, so no loop
                # 3) if the end moving direction (dx, dy) from start point (x, y) can connect to the possible starting direction of the new point (nx, ny)
                if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in visited and (dx, dy) in type_to_direction[grid[nx][ny]]:
                    q.append((nx, ny))
                    visited.add((nx, ny))
        return False

<b> 1743. Restore the Array From Adjacent Pairs </b>

construct a hashmap as a graph, key is element, value is the list of elements (max 2) that it has connection, no order  
then loop over the graph to find the first element which only has one edge, that is the start point. note that there will be only 2 such elements, one is start, one is the ending point, whichever is good.

keep a set for visited items and a queue for updating what to check next


time complexity 
-   O(N), linearly loop over the elements
-   For a valid linear array of N unique elements, there are exactly N − 1 adjacent pairs.

In [9]:

from collections import defaultdict, deque
def restoreArray(adjacentPairs):
    """
    :type adjacentPairs: List[List[int]]
    :rtype: List[int]
    """
    # construct a hashmap indicating the graph using
    # every element in the pairs, and find the first element which only has one pointing out element
    # that is the start point, note that there can be two such elements (one start point one end point)
    # since it is guaranteed that the adjacent pairs and integer array is valid,
    # no need additional validation.

    graph = defaultdict(list)
    for u, v in adjacentPairs:
        graph[u].append(v)
        graph[v].append(u)
    # find the first key whose value only has length 1
    start = None
    for k, list_of_v in graph.items():
        if len(list_of_v) == 1:
            start = k
            break
    result = [start]
    visited = {start}
    queue = deque(graph[start])
    while queue:
        item = queue.popleft()
        if item not in visited:
            result.append(item)
            visited.add(item)
            for i in graph[item]:
                queue.append(i)
    return result


In [10]:
adjacentPairs = [[2,1],[3,4],[3,2]]
restoreArray(adjacentPairs)

[1, 2, 3, 4]

<b> Robinhood question: Referral count </b>

Robinhood is famous for its referral program. It’s exciting to see our users spreading the word across their friends and family. One thing that is interesting about the program is the network effect it creates. We would like to build a dashboard to track the status of the program. Specifically, we would like to learn about how people refer others through the chain of referral.

For the purpose of this question, we consider that a person refers all other people down the referral chain. For example, A refers B, C, and D in a referral chain of A -> B -> C -> D. Please build a leaderboard for the top 3 users who have the most referred users along with the referral count.

Referral rules:

A user can only be referred once.
Once the user is on the RH platform, he/she cannot be referred by other users. For example: if A refers B, no other user can refer A or B since both of them are on the RH platform.
Referrals in the input will appear in the order they were made.
Leaderboard rules:

The user must have at least 1 referral count to be on the leaderboard.
The leaderboard contains at most 3 users.
The list should be sorted by the referral count in descending order.
If there are users with the same referral count, break the ties by the alphabetical order of the user name.

input: [["A", "B"], ["A", "C"], ["C", "D"], ["E", "F"],["E", "G"]]  
middle: {"A": 3, "B": 0, "C" 2, "D": 0, "E": 2, "F": 0, "G": 0}  
output: ["A", "C", "E"]

In [16]:
from collections import defaultdict
from typing import List
def getNumberOfBacklogOrders(referrals:List[List[str]]):
    # construct a graph, like {"A": ["B", "C"], "B": ["D"]}
    graph = defaultdict(list)
    for refer, friend in referrals:
        graph[refer].append(friend)

    # store the refer count map like {"A": 3, "B": 1}
    count_map = defaultdict(int)
    def dfs_helper(node):
        if len(graph[node]) == 0:
            count_map[node] == 0
            return 0
        if node in count_map:
            return count_map[node]
        count = 0
        for sub_node in graph[node]:
            count += dfs_helper(sub_node) + 1
        count_map[node] = count
        return count
    for item in list(graph.keys()):
        dfs_helper(item) # populate the count_map
    print(count_map)
    return sorted(count_map, key=lambda x: (-count_map[x], x))[:3]



In [4]:
"""
N: number of users
graph construction and the len counting both take O(N)
sorting takes O(N log N)
so final time complexity is O(N log N)
"""
from collections import defaultdict
from typing import List
def getNumberOfBacklogOrders(referrals:List[List[str]]):
    # construct a graph, like {"A": ["B", "C"], "B": ["D"]}
    graph = defaultdict(list)
    for refer, friend in referrals:
        graph[refer].append(friend)
    
    graph_count = {node: len(graph[node]) for node in graph}
    return sorted(graph_count, key=lambda x: (-graph_count[x], x))[:3]

In [5]:
getNumberOfBacklogOrders( [["A", "B"], ["A", "C"], ["C", "D"], ["E", "F"],["E", "G"]])

['A', 'E', 'C']

In [17]:
getNumberOfBacklogOrders( [["A", "B"], ["A", "C"], ["C", "D"], ["E", "F"],["E", "G"]])

defaultdict(<class 'int'>, {'B': 0, 'D': 0, 'C': 1, 'A': 3, 'F': 0, 'G': 0, 'E': 2})


['A', 'E', 'C']

In [18]:
a = {"D":0, "E" :2, "A":3, "B": 2}
sorted(a, key=lambda x: (-a[x], x))

['A', 'B', 'E', 'D']

In [19]:
getNumberOfBacklogOrders([["A", "B"], ["A", "C"], ["C", "D"]])

defaultdict(<class 'int'>, {'B': 0, 'D': 0, 'C': 1, 'A': 3})


['A', 'C', 'B']

### <b> Dijkstra's algorithm </b>

finding the shortest paths between nodes in a weighted graph. The algorithm uses a <b>min-priority queue</b> data structure for selecting the shortest paths known so far.

<b> 787 cheapest flight within k stops </b>

option 1: using normal queue and keep track of k to keep the valid next node, however, this way exceeds memory limitation

example input: 
n = 17. 
flights = [[0,12,28],[5,6,39],[8,6,59],[13,15,7],[13,12,38],[10,12,35],[15,3,23],[7,11,26],[9,4,65],[10,2,38],[4,7,7],[14,15,31],[2,12,44],[8,10,34],[13,6,29],[5,14,89],[11,16,13],[7,3,46],[10,15,19],[12,4,58],[13,16,11],[16,4,76],[2,0,12],[15,0,22],[16,12,13],[7,1,29],[7,14,100],[16,1,14],[9,6,74],[11,1,73],[2,11,60],[10,11,85],[2,5,49],[3,4,17],[4,9,77],[16,3,47],[15,6,78],[14,1,90],[10,5,95],[1,11,30],[11,0,37],[10,4,86],[0,8,57],[6,14,68],[16,8,3],[13,0,65],[2,13,6],[5,13,5],[8,11,31],[6,10,20],[6,2,33],[9,1,3],[14,9,58],[12,3,19],[11,2,74],[12,14,48],[16,11,100],[3,12,38],[12,13,77],[10,9,99],[15,13,98],[15,12,71],[1,4,28],[7,0,83],[3,5,100],[8,9,14],[15,11,57],[3,6,65],[1,3,45],[14,7,74],[2,10,39],[4,8,73],[13,5,77],[10,0,43],[12,9,92],[8,2,26],[1,7,7],[9,12,10],[13,11,64],[8,13,80],[6,12,74],[9,7,35],[0,15,48],[3,7,87],[16,9,42],[5,16,64],[4,5,65],[15,14,70],[12,0,13],[16,14,52],[3,10,80],[14,11,85],[15,2,77],[4,11,19],[2,7,49],[10,7,78],[14,6,84],[13,7,50],[11,6,75],[5,10,46],[13,8,43],[9,10,49],[7,12,64],[0,10,76],[5,9,77],[8,3,28],[11,9,28],[12,16,87],[12,6,24],[9,15,94],[5,7,77],[4,10,18],[7,2,11],[9,5,41]]  
src = 13,  
dst = 4,  
k = 13.  

using K>0 to prune and shorten the queue is not enough, should using min_price as another guard to prune the valid item added to the queue.

BFS at the beginning, the minPrice is inf at the beginning, so add all next nodes, until the first route that hits the desitation, the min_price get updated, the Q will selectively adding new next nodes that can produce less pricy route, otherwise just finish the rest of the option in existing Queue.

time complexity: 
-   BFS based, suppose N: number of cities (node), E: number of flights (edge)
-   bulding graph: O(E)
-   queue can grow large, in worse case for every city, it can be visited K+1 times, so unique pair of (city, stop) is O(NK). for each city it can hae up to E outgoing flights, so in total it is O(NKE)
-   worse case the time complexity is O(NKE)

space complexity:
-   graph: O(E)
-   queue: O(NK) since each (node, level) can be in the queue
-   in total O(E+NK)

In [ ]:
from collections import defaultdict, deque
from typing import List
class Solution:
    # after adding 2 ways of guard: k value, and min_price, to prune the graph, still get memory exceed limit error
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # generate a graph in dict
        flights_graph = defaultdict(dict)
        for s, e, p in flights:
            flights_graph[s][e] = p

        # construct a queue
        q = deque([(0, src, k+1)])
        min_price = float('inf')
        while q:
            price, node, level = q.popleft()
            if node == dst:
                min_price = min(min_price, price)
            if level > 0:
                for next_node in flights_graph[node]:
                    if price + flights_graph[node][next_node] < min_price:
                        q.append((price + flights_graph[node][next_node], next_node, level-1))
        return min_price if min_price != float('inf') else -1






using the Dijkstra's algorithm to solve the problem.  
time complexity:  
-   V: number of nodes, E: number of edges  
-   O(E) E is the number of edges (flights) to construct the graph  
-   priority queue, worse case each edge (flight) gets pushed to the heapq, so insertion times (the while loop) -> O(E)
 -   however the queue itself averagely has length O(V)  -> due to the Dijkstra's algorithm, just it can grow into O(E) in worse case. each heap operation: O(logE)  for E times -> O(ElogE)

space complexity:   


In [20]:
from collections import defaultdict
from heapq import heappush, heappop

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # generate a graph in dict
        flights_graph = defaultdict(dict)
        for s, e, p in flights:
            flights_graph[s][e] = p

        # construct a priority queue
        q = [(0, src, k+1)] # price, city, level
        while q:
            price, node, level = heappop(q)
            if node == dst:
                # the first node from priority queue that reach the desitination has the lowest price
                # because its child path will have higher price than itself
                # and the rest of the paths in the queue will have the higher price too because it is priority queue
                # same, the child path (maybe havent visited yet) of the rest of the paths in the queue will have higher price then its parent, and
                # of course higher than the first path in the queue.
                # so the first one reach the desition has the lowest price
                return price
            if level > 0:
                # using level to keep track of the valid path.
                # if exceeding the level, then no need to consider it.
                for next_node, p in flights_graph[node].items():
                    heappush(q, (price + p, next_node, level-1))
        return -1



In [21]:
a = Solution()
a.findCheapestPrice(4, [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]], 0, 3, 1)

700

using additional list to tract if a node is already visited previously, if yes, skip that path. This bring down the time complexity.

<b>time complexity</b>: 

O(ELogE), as the same as previous option, but using vis should improve the complexity

- searching in a heap queue is a bineary search, so each iteration uses O(logN) where N is the number of elements of that queue.
- graph construction: O(E) where E is the number of edge
- each heappush and heappop takes O(logV) where V is the number of nodes. in the worse case, a city will be visited multiple times (up to k+1 times), and for each visit, its outgoing edges are processed. so E edges and k+1 possible visit per node, results in O((K+1)E*LogV)
- O(E) + O((K+1)E*LogV) = O(K*E*LogV) -> this is the worse cases time complexity.
    - each edge E can cause a vertex to be added to the heap k+1 times (once for each allowed level), and LogV is the time complexity for each heap queue operation
- the vis list helps to reduce the repeated processing of a node visited before, so avg and best case time complexity should be improved.

Why each heap operation has time complexity of O(LogV) where V is the number of cities, instead of O(LogE) where E is the number of edges?

- Although the total number of operations is influenced by the edges (since each edge may cause a node to be added to the heap), the complexity of each individual heap operation is determined by the number of elements in the heap, and the heap contains at most V nodes at any given time. -> extension wont happen to add all edges to the heap at one time

<b> space complexity </b>  
Graph: O(E) where E is the number of edges  
priority Q: max num of elements in the Q can be portiional to the num of edges because each edge might add a new element to the queue, so O(E)  
visited list: O(V) V is the num of nodes / cities

so total: O(E)+O(E)+O(V) = O(E) because E is generally larger than V so space complexity is dominated by O(E)

In [22]:
from collections import defaultdict
from heapq import heappush, heappop

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # generate a graph in dict
        flights_graph = defaultdict(dict)
        for s, e, p in flights:
            flights_graph[s][e] = p

        # construct a priority queue
        q = [(0, src, k+1)] # price, city, level
        vis = [0] * n
        while q:
            price, node, level = heappop(q)
            if node == dst:
                # the first node from priority queue that reach the desitination has the lowest price
                # because its child path will have higher price than itself
                # and the rest of the paths in the queue will have the higher price too because it is priority queue
                # same, the child path (maybe havent visited yet) of the rest of the paths in the queue will have higher price then its parent, and
                # of course higher than the first path in the queue.
                # so the first one reach the desition has the lowest price
                return price
            if vis[node] >= level:
                # if this node is visited before, skip this node
                # when level is larger, this node is visted in higher level, so visited previously
                continue
            vis[node] = level
            if level > 0:
                # using level to keep track of the valid path.
                # if exceeding the level, then no need to consider it.
                for next_node, p in flights_graph[node].items():
                    heappush(q, (price + p, next_node, level-1))
        return -1

if want to also output the actual path, can add an additional list to store the path

In [23]:
from collections import defaultdict
from heapq import heappush, heappop
from typing import List, Tuple

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> Tuple[int, List[int]]:
        # Generate the graph in dict
        flights_graph = defaultdict(dict)
        for s, e, p in flights:
            flights_graph[s][e] = p

        # Construct the priority queue
        q = [(0, src, k+1, [src])]  # price, city, level, path
        vis = [0] * n
        while q:
            price, node, level, path = heappop(q)
            if node == dst:
                # The first node from the priority queue that reaches the destination has the lowest price
                # because its child path will have a higher price than itself
                # and the rest of the paths in the queue will have a higher price too because it is a priority queue.
                # Similarly, the child path (maybe not visited yet) of the rest of the paths in the queue will have a higher price than its parent, and
                # of course higher than the first path in the queue.
                # So the first one to reach the destination has the lowest price.
                return price, path
            if vis[node] >= level:
                # If this node is visited before, skip this node
                # When the level is larger, this node is visited at a higher level, so it was visited previously.
                continue
            vis[node] = level
            if level > 0:
                # Using level to keep track of the valid path.
                # If exceeding the level, then no need to consider it.
                for next_node, p in flights_graph[node].items():
                    heappush(q, (price + p, next_node, level - 1, path + [next_node]))
        return -1, []

In [24]:
a = Solution()
a.findCheapestPrice(4, [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]], 0, 3, 1)

(700, [0, 1, 3])

#### Eulerian Cycle 
a cycle that visits every edge in a graph exactly once and starts and ends at the same vertix.

#### Hierholzer's algorithm
[link](https://en.wikipedia.org/wiki/Eulerian_path#Hierholzer's_algorithm).  
an algorithm to find the euler cycle



<b> 2097 valid arrangement of pairs </b>

for selecting start point, initializing it with random valid point from the pairs, because when all the vertex are connected, the degree for each of the vertex is 0, and wont be able to find the 1 value vertex.

for the degree map, 1 means the entry point, the end will have value -1, middle vertex has value 0  

time complexity: loop over pairs once to construct 2 dictionary, the time complexity is O(N). when using the dfs helper function to find the path, the graph is visited once only, so time complexity is also O(N). in total time complexity is O(N). N is the size of pairs, or the edge number


In [ ]:
from collections import defaultdict
from typing import List
def validArrangement(pairs: List[List[int]]) -> List[List[int]]:
    # hierholzer's algorithm to find euler cycle
    graph = defaultdict(list)
    degree = defaultdict(int)
    # construct 2 dictionary, one as graph, another as degree dict to find the starting point
    for x, y in pairs:
        graph[x].append(y)
        degree[x] += 1
        degree[y] -= 1

    # the problem assumes that there is always a valid arrangement, so find the start point when the degree is not 0
    # randomly picking up starting point, in case the vertex is connected, the degree will all be 0, for example: [[1,3],[3,2],[2,1]]
    start = pairs[0][0]  
    for k in degree:
        if degree[k] == 1: # 1 means the entry point, the end will have value -1, middle vertex has value 0
            start = k
            break
    res = []
    # the last item is added first so need to reverse.
    def helper(x):
        # example that needs this while loop: [[1, 2], [1, 3], [2, 3], [3, 1]]
        while graph[x]: 
            helper(graph[x].pop())
        res.append(x)

    helper(start)
    res.reverse()
    return [[res[i], res[i+1]] for i in range(len(res)-1)]

In [27]:
validArrangement([[1,3],[3,2],[2,1]])

[[1, 3], [3, 2], [2, 1]]

In [7]:
validArrangement([[1, 2], [1, 3], [2, 3], [3, 1]])

[[1, 3], [3, 1], [1, 2], [2, 3]]

<b> 399 evalute division </b>

Graph problem with DFS helper.  
first construct a graph representing the equation. Then for each of the queries, using x as the starting point, try to find a node with value y, then update the cumulative value.

time complexity:
-   N: num of equations, Q: number of queries. V: number of distinct values, E: number of edges, E = 2N = O(N)
- constructing the graph: O(E) = O(N)
- for each query item, it will call the helper(x, y) function, which in worse case will loop over all the nodes and check all edges to find the solution. so totally O(Q*(V+E))
    - why worse case is O(V) + O(E) per query instead of just O(E): the helper function check if the node is visted or not, worse case it checked all nodes, then check the edge of this node.
    - each node is visited once in visited set, each edge is explored once 
    -   for DFS, each node you visited, you mark it as visited which cost O(1), then you iterate over all its neighbors, which depends on the node's degree. so cost of DFS = (work per node) + (work per edge). in the outer while loop, for each loop, it is O(1) + O(some edge), for whole loop, when it adds up, it is O(V)+O(E)
- in total O(N) + O(Q*(V+E)) = O(N) + O(Q*(V+N)), V << N, so O(Q*N)

In [1]:
from collections import defaultdict
from typing import List
def calcEquation(equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
    result = [-1.0]*len(queries)
    # construct the graph
    graph = defaultdict(dict)
    for ind, (i, j) in enumerate(equations):
        graph[i][j]= values[ind]
        graph[j][i]= 1/values[ind]

    def helper(x, y):
        # find path from x to y, given a graph
        queue = [(x,1)]
        visited = set()
        while queue:
            node, val = queue.pop() # not a deque, so DFS, not BFS
            if node in visited:
                continue
            visited.add(node)
            neighbors = graph[node] # a dict {node:value}
            if y in neighbors:
                return val*neighbors[y]
            else:
                for n in neighbors:
                    queue.append((n, val*neighbors[n]))
            # queue = next_queue
        return -1.0

    for ind, (x, y) in enumerate(queries):
        if x not in graph or y not in graph:
            continue
        elif x==y and x in graph[x]:
            result[ind] = 1.0
        elif y in graph[x]:
            result[ind] = graph[x][y]
        else:
            # find the transition path
            # there maybe multiple paths, but since the assumption is there is no contradiction, it is safe to pick random one
            # depth first search, search neighbors and neighbors' neighbors until get the target

            result[ind] = helper(x, y)
    return result

In [31]:
equations = [["a","b"],["b","c"],["bc","cd"]]
values = [1.5,2.5,5.0]
queries = [["a","c"],["c","b"],["bc","cd"],["cd","bc"]]
print(calcEquation(equations, values, queries))

[3.75, 0.4, 5.0, 0.2]


Adding a cache layer to store the intermediate value, for example: if a -> b -> c -> d path is visited, next time when query the a -> c, the result will be retrieved from cache directly.

Time complexity:
-   worse case (all query are not related), still the same as previous solution, O(Q(V+N))
-   on average it should be faster, because the cache graph becomes nearly fully connected.

space complexity:
-   increase due to the cache layer, worse case it is O(V*V), for every node, connect to other node, both direction, V*(V-1) = O(V*V)

In [ ]:
# solution with cache to reduce the time complexity
from typing import List, Dict, Tuple
def calcEquation(equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
    graph = defaultdict(dict)
    for ind, (i, j) in enumerate(equations):
        graph[i][j]= values[ind]
        graph[j][i]= 1/values[ind]

    cache: Dict[Tuple[str, str], float] = {}
    def dfs_helper(x: str, y: str) -> float:
        if (x, y) in cache:
            return cache[(x, y)]
        queue = [(x, 1.0)]
        visited = set()
        mid_val = {x: 1.0} # key: node, value: value from x starting node to this intermediate node
        
        while queue:
            node, val = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            if node == y:
                for mid, v in mid_val.items():
                    cache[(x, mid)] = v
                    cache[(mid, x)] = 1/v
                return val
            for nei, v in graph[node].items():
                if nei not in visited:
                    mid_val[nei] = val*v
                    queue.append((nei, val*v))
        cache[(x, y)] = -1.0
        return -1.0
    results = [-1.0]*len(queries)
    for i, (x, y) in enumerate(queries):
        if x not in graph or y not in graph:
            continue
        elif x==y and x in graph[x]:
            cache[(x, y)] = 1.0
            results[i] = 1.0
        elif y in graph[x]:
            cache[(x, y)] = graph[x][y]
            results[i] = graph[x][y]
        else:
            results[i] = dfs_helper(x, y)
    return results

<b> 863 all nodes K distance in binary tree </b>

actuall a tree to graph problems. first transform the tree into a graph: node -> list of connected nodes, then using for loop to find the kth level nodes

time complexity:
- building graph, visited each node once O(N)
- BFS visit, for k level, worse  case visit all node once, O(N)
- total O(N)

space complexity:
- graph storage: for tree with N node, the edge is N-1, for undirected graph it is double the edge still O(N)
- visited and queue at most holds all the nodes so O(N)
- recurisve space in graph creating is O(H) H is the tree hight, in worse case H is N, in balanced tree it is O(logN)
- in total still O(N)

In [3]:
from collections import defaultdict
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None

class Solution:
    def distanceK(self, root: TreeNode, target: TreeNode, k: int) -> List[int]:
        """
        step 1:
        transform the tree into a graph:
        node: [list of its parent or children nodes]
        step 2:
        starting from the target node, with level 0, visit its connected node from the graph, with a visited set to avoid duplicated visiting. dfs search until the level reaching k
        assumption: node value in the tree is unique
        """
        graph = defaultdict(list)

        def to_graph(parent, child):
            nonlocal graph
            if parent and child:
                graph[parent.val].append(child.val) 
                graph[child.val].append(parent.val) 
            if child.left:
                to_graph(child, child.left)
            if child.right:
                to_graph(child, child.right)

        to_graph(None, root) # smart to use None as "parent" of root
        if not root:
            return []
      
        queue = [target.val]
        visited = {target.val}
        for _ in range(k):
            # loop k times to reach the kth level
            new_queue = [] # each level has a queue of nodes
        # graph stores parent -> child, child -> parent, so possible to visit duplicated node
            for node in queue:
                # only add unvisited node to the next level
                for conn_node in graph[node]:
                    if conn_node not in visited:
                        new_queue.append(conn_node)
                visited.add(node)
            # when the loop finished, queue actually contains nodes at kth level 
            queue = new_queue
        return queue



In [3]:
from collections import defaultdict, deque
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None

class Solution:
    def distanceK(self, root: TreeNode, target: TreeNode, k: int) -> List[int]:
        """
        step 1:
        transform the tree into a graph:
        node: [list of its parent or children nodes]
        step 2:
        starting from the target node, with level 0, visit its connected node from the graph, with a visited set to avoid duplicated visiting. dfs search until the level reaching k
        assumption: node value in the tree is unique
        """
        graph = defaultdict(list)

        def to_graph(node, parent):
            # this recursive func is easier to understand than previous one
            nonlocal graph
            if not node: return
            if parent: graph[node.val].append(parent.val)
            if node.left:
                graph[node.val].append(node.left.val)
                to_graph(node.left, node)
            if node.right:
                graph[node.val].append(node.right.val)
                to_graph(node.right, node)
            
        to_graph(root, None) # smart to use None as "parent" of root
        if not root:
            return []

        queue = deque([(target.val, 0)])
        visited = set()
        res = []
        while queue:
            node_val, level = queue.popleft()
            if node_val in visited: continue
            visited.add(node_val)
            if level == k:
                res.append(node_val)
            elif level < k:
                for nei in graph[node_val]:
                    queue.append((nei, level+1))
        
        return res



<b> 1443 minimium time to collect all apples in a tree </b>

the tree is undirected, so have to create a graph with bidirection. The tree is not circular. But when checking the node, need to ensure it is not its parent node, to avoid going backward.

the dfs helper goes to the leave node first, then moving up, +2 only when the node itself has an apple or it has children node that has apple.

using dfs to add up children's apple count eliminate the issue of double counting the path back to the original root node.

why undirected graph? because not sure if the provided edges list is in right order, like [0, 1] or [1, 0]

In [4]:
from collections import defaultdict

def minTime(n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
    """
    result is always even number because you return to the original node in the same path back
    dfs starting from root vertice, current node result + child node result, in this way it does not count the path back to parent vertice duplicatedly 
    """
    graph = defaultdict(list) # undirected graph
    for i, j in edges:
        graph[j].append(i) 
        graph[i].append(j)

    def helper(node, parent):
        res = 0
        # post order
        for n in graph[node]:
            if n != parent: # not going backward
                res += helper(n, node)

        # base case:
        # 1) node has children with apple
        # 2) node itself has an apple
        # 3) root node no need to add count 2, already at root
        if node!=0 and (res or hasApple[node]):
            res += 2
        return res

    return helper(0, -1)

In [6]:
minTime(4, [[0,2],[0,3],[1,2]], [False,True,False,False])

4

<b> 269 alien dictionary </b>

Lexicographically Smaller:  
A string a is lexicographically smaller than a string b if in the first position where a and b differ, string a has a letter that appears earlier in the alien language than the corresponding letter in b.  
If the first min(a.length, b.length) characters do not differ, then the shorter string is the lexicographically smaller one. 

first construct the dictionary of char -> [smaller char], and a reverse_dictionary to track the number of parent to the char. 

When a char in reverse_dictionary has 0 parent, that means its order is now the highest.



In [ ]:
from collections import defaultdict, deque

def alienOrder(words: List[str]) -> str:
    """
    topologically sorting a graph
    """
    graph = defaultdict(set)
    reverse_graph = defaultdict(int)
    # construct the graph of char -> [child char]
    # construct reverse graph of incoming node count char: parent_count
    # constrct distinct chars using reverse_graph, even though the initialized value 0 is not necessary
    for s in words:
        for c in s:
            reverse_graph[c] = 0

    if not words or len(words) == 0: return ""
    # comparing two adjacent words, the first char different from the two words reflect the lexicographical order.
    # wef, wez -> f > z, but cannot say w > e
    for i in range(len(words)-1):
        cur_w = words[i]
        next_w = words[i+1]
        l = min(len(cur_w), len(next_w))
        for j in range(l):
            char_1 = cur_w[j]
            char_2 = next_w[j]
            if char_1 != char_2:
                if char_2 not in graph[char_1]:
                    graph[char_1].add(char_2)
                    reverse_graph[char_2] += 1
                # as long as the first different char is in the order, the follow up chars order do not matter, for example  aac > aba because at 2nd char a>b, even at 3rd char c<a
                break
            # if word 1 is longer than word 2, but word2 is word1's prefix, word 2 should be smaller than word 1, however the order is word 1 before word 2, which is invalid
            # at this stage cur_w and next_w are the same up to length l
            if j == l-1 and len(cur_w) >  len(next_w):
                return ""

    queue = deque([c for c in reverse_graph if reverse_graph[c]==0])
    res = []
    # find the char that does not incoming edge, which means they have higher order right now
    while queue:
        c = queue.popleft()
        res.append(c)
        for child in graph[c]:
            reverse_graph[child] -= 1
            if reverse_graph[child] == 0:
                queue.append(child)

    if len(res) < len(reverse_graph.keys()): return ""
    return "".join(res)

In [9]:
alienOrder(["abc", "ab"])

''

<b> 797 all paths from source to target </b>

using DFS instead of backtracking for this problem because no cyclic exist in a single directed graph, 0 is start, N is end

time complexity
-   worse case, the num of possible paths from node 0 to node N-1 is 2^(N-2): for N nodes, except 0 and N-1, the N-2 nodes in between has 2 option, either included in the path or not, so 2^(N-2)
-   for each path, construting the path (copying list) takes O(N)
-   in total: O(2^(N-2))

space complexity
-   result list: having O(2^(N-2)) result, each path can be at most size N, so O(N*2^(N-2))
-   recursion stack can go as deep as O(N) for inbalanced tree.

In [ ]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        """
        memo may not save too much time
        worse case edge count is 2^(N-2)
        """
        def dfs(node, path):
            if node == len(graph)-1:
                res.append(path[:])
            else:
                for neighbor in graph[node]:
                    dfs(neighbor, path+[neighbor])
        res = []
        dfs(0, [0])
        return res

In [ ]:
from collections import defaultdict
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        """
        time complexity:
        -   graph traversal depends on how many paths exist. 
        in worse case for N nodes, the maximum path number is 2^(N-2) -2 
        because 0 and n must be in the path, so the leftover N-2 nodes would have 
        the choice either be in the path or not in the path.
        -   O(2^(N-2) * V) V is the average cost for constructing the path
        """
        n = len(graph)
        stack = [(0, [0])] # (node, [path])
        res = []
        while stack:
            node, path = stack.pop()
            # reach the end
            if node == n-1:
                res.append(path)
            for neighbor in graph[node]:
                stack.append((neighbor, path+[neighbor]))
        return res


<b> 277 find the celebrity </b>

n people labled from 0 to n-1, there may exist 1 celebrity that all other ppl knows it but it does not know others. a knows(a, b) function is given to check if a knows b, return bool.

time complexity:
- first loop finds the candidate celebrity. O(N)
-  then validate if this celebrity is really the celebrity, it uses another for loop so another O(N)
-   here we don know what is the time complexity of the given function knows, so did not include it
- in total it is O(N).

However if the knows() func is time consuming, adding the @lru_cache will be helpful.

space complexity
- O(1) if not using cache, otherwise O(E) E is the edge size

In [ ]:
# The knows API is already defined for you.
# return a bool, whether a knows b
# def knows(a: int, b: int) -> bool:
from functools import lru_cache

def knows(a: int, b: int) -> bool:
    pass

class Solution:
    @lru_cache(maxsize=None)
    def cachedKnows(self, a, b):
        # cache the result of knows call
        return knows(a, b)

    def findCelebrity(self, n: int) -> int:
        """
        a celebrity is the node in the graph that has n-1 directed
        edges going in (everybody knows it), 
        but no edge going out (dont know anybody)

        if a knows b, rule out a, if a does not know b, rule out b -> loop n until 1 candidate left
        then validate if this is true one because n-1 should know it and it should know nobody.
        first find the candidate, then validate if it is truely the celebrity

        """
        self.n = n
        celebrity_candidate = 0
        for i in range(1, n):
            if self.cachedKnows(celebrity_candidate, i):
                # change to another candidate
                celebrity_candidate = i
        
        if self.is_celebrity(celebrity_candidate):
            return celebrity_candidate
        return -1
    def is_celebrity(self, i):
        for j in range(self.n):
            if i == j: continue
            # i knows someone, or someone does not know i, then i
            # is not qualified as a celebrity
            if self.cachedKnows(i, j) or not self.cachedKnows(j, i):
                return False
        return True

<b> 886 possible bipartition </b>

actually similar to island flood problems. Here using two colors to distinguish two group. If found any two nodes with conflicted color, then the separation is not successful.

The goal is to separate nodes with edge into two different colors. If during the loop finding any two nodes with edge and with same color then the separation is not successful.

node is initialized with no color, meaning not visited, then assign an initial color, and visit its neighbor, if not visited, assignining oppositive color, if with color, check color compatibility.


BFS solution
- time complexity:
    -   each node is visited once by adding / popping out of queue, so O(V)
    -   each edge is visited once through the graph, so O(E)
    -   in total O(V+E)
- space complexity:
    -   graph construction: O(E) related to the edge count
    -   color list: O(V) related to number of node
    -   queue length no more than the V
    -   in total O(V+E)

In [5]:
from collections import defaultdict, deque
class Solution:
    def possibleBipartition(self, n: int, dislikes: List[List[int]]) -> bool:
        """
        use two color to separate the group
        if two nodes with edge end up with same color, then there is conflict,
        cannot split into two groups
        BFS: start with node with no color, then assign color, check its neighbor
        if the neighbor does not have color, assign a proper one and put to queue
        if neighbor has conflicted color, then directly return false
        """
        def bfs(start_node):
            # node is entering node with no color
            q = deque([start_node])
            color[start_node] = 0 # initialize it with color

            while q:
                node = q.popleft()
                for nei in graph[node]:
                    # color is used as visited set already
                    if color[nei] == color[node]: return False
                    if color[nei] == -1:
                        # assign opposite color
                        color[nei] = 1 - color[node]
                        # check this new node, no color means not checked before
                        q.append(nei)
            return True


        graph = defaultdict(list)
        for i, j in dislikes:
            graph[i].append(j)
            graph[j].append(i)
        color = [-1]*(n+1) # initiaize with no color
        for i in range(1, n+1):
            if color[i] == -1:
                # very much like the island flood problems
                if not bfs(i): return False
        return True

In [6]:
dislike = [[1,2],[1,3],[2,4]]
n = 4
s = Solution()
s.possibleBipartition(n, dislike)          

True

same as using DFS way. Below are two ways of doing the DFS implementation
- one is to change the queue into a normal list instead of a deque
- another is to use recursive inside of the function

In [ ]:
# graph = defaultdict(list)
# for i, j in dislikes:
#     graph[i].append(j)
#     graph[j].append(i)
# color = [-1]*(n+1) # initiaize with no color

def dfs(start_node):
    # node is entering node with no color
    q = [start_node] # => here, just a normal queue then everytime newly added item is popped first, so DFS
    color[start_node] = 0 # initialize it with color

    while q:
        node = q.pop()
        for nei in graph[node]:
            # color is used as visited set already
            if color[nei] == color[node]: return False
            if color[nei] == -1:
                # assign opposite color
                color[nei] = 1 - color[node]
                # check this new node, no color means not checked before
                q.append(nei)
    return True

def dfs2(node, node_color):
    color[node] = node_color
    for nei in graph[node]:
        if color[nei] == node_color: return False
        # if neighbor has not been colored, but color it with opposite one has conflicts
        if color[nei] == -1 and not dfs2(nei, 1 - node_color):
                return False
    return True

<b> 851 Loud and Rich </b>

among the people who is reacher than i, find the ppl with smallest number in quite array, including ppl itself as well.

time complexity
-   O(len(richer_array)) for graph construction
-   O(N) for interating N people only once.
-   the helper function only visit the graph once, which is lated to len(richer_array) as well
-   so total is O(N) + O(Len(richer_array)) = O(N+E) E means edge number of the graph

space complexity:
-   graph size O(E)
-   result array O(N)
-   dfs helper call stack O(N) in worse case
-   in total O(N+E)


In [16]:
from collections import defaultdict

def loudAndRich(richer: List[List[int]], quiet: List[int]) -> List[int]:
    # find ppl who are richer than i but with the smallest quite value, including i itself
    n = len(quiet)
    res = [-1]*n
    def helper(i):
            # find the right person and update res array at the same time

        if res[i] >= 0: 
            # if visited before, directly return the result
            return res[i]
        res[i] = i
        for next_i in graph[i]:
            res_for_next_i = helper(next_i)
            if quiet[res[i]] > quiet[res_for_next_i]:
                res[i] = res_for_next_i
        return res[i]

    graph = defaultdict(list)
    for a, b in richer:
        graph[b].append(a)
    for i in range(n):
        helper(i)
    return res

In [17]:
loudAndRich([[1,0],[2,1],[3,1],[3,7],[4,3],[5,3],[6,3]], [3,2,5,4,6,1,7,0])

[5, 5, 2, 5, 4, 5, 6, 7]

<b> 3481 apply substitutions </b>

You are given a replacements mapping and a text string that may contain placeholders formatted as %var%, where each var corresponds to a key in the replacements mapping. Each replacement value may itself contain one or more such placeholders. Each placeholder is replaced by the value associated with its corresponding replacement key.

Return the fully substituted text string which does not contain any placeholders.

Input: replacements = [["A","abc"],["B","def"]], text = "%A%_%B%"

Output: "abc_def"

Input: replacements = [["A","bce"],["B","ace"],["C","abc%B%"]], text = "%A%_%B%_%C%"

Output: "bce_ace_abcace"

text can also be "%A%c%B%" -> only A, B are valid key

In [18]:
# below is my original solution, not too optimal
"""
the idea is to keep all sub portion of text that is not comleted in a deque, then keep updating it until no % is found
but the implementation is a bit complicated, compared to the better solution below with simiar idea
"""
import re
from collections import deque
class Solution:
    def applySubstitutions(self, replacements: List[List[str]], text: str) -> str:
        def is_complete(s):
            # return boolean
            pattern = r"%(\w)%"
            return not re.search(pattern, s)
        
        def re_helper(s):
            pattern = r"%(\w)%"
            matches = re.finditer(pattern, s)
            return matches

        def generate_new_v(v, graph):
            new_v = []
            i = 0
            matches = re_helper(v)
            for m in matches:
                k = m.group(1)
                ind = m.start(0)
                new_v.append(v[i:ind])
                if k in graph:
                    new_v.append(graph[k])
                else:
                    new_v.append(v[ind: ind+3])
                i = ind+3
            new_v.append(v[i:])
            new_v_str = "".join(new_v)
            return new_v_str

        graph = {}
        q = deque([])
        # construct graph and q
        for k, v in replacements:
            is_complete_flag = is_complete(v)
            if is_complete_flag:
                graph[k] = v
            else:
                q.append([k, v])

        while q:
            k, v = q.popleft()
            new_v_str = generate_new_v(v, graph)
            if is_complete(new_v_str):
                graph[k] = new_v_str
            else:
                q.append([k, new_v_str])
        res = generate_new_v(text, graph)
        return res


this is the better solution with similar idea above. It uses the split function to find out which substring should be kept (always the even position one), and which substring is the key (always the odd position). It keep updating the text with the replacements value, until no % is found

time complexity
- M: len of replacement, D: max depth of nested substituions, N: len of final expanded text after all substitutions
-   building graph takes O(M)
-   main while loop: in worse case, each substitution can introduce new placeholders. each pass split and join the text, which is O(L) L is the current size of the text str. the number of while loop iteration is at most the max nesting depth D in worse case (each iteration solves one layer of D) -> so total time is O(D*L) -> O(D*N)
- in total O(D*N + M). when no deep nesting D is small, then the complexity will be smaller too.


space complexity
-   graph O(M)
-   intermediate string inside of main while loop: each generate a new string and split array, with length up to N, but not cumulative, only live within that iteration.
-   so total O(M+N)

In [ ]:
import re
from collections import deque
class Solution:
    def applySubstitutions(self, replacements: List[List[str]], text: str) -> str:
        graph = {}
        for k, v in replacements:
            graph[k] = v
        while "%" in text:
            new_str = []
            text_split = text.split("%")
            # example: "_%A%_%B%__".split("%") -> ['_', 'A', '_', 'B', '__']
            # "%A%_%B%".split("%") -> ['', 'A', '_', 'B', ''] same length
            # "%A%b%C%".split("%") -> ['', 'A', 'b', 'C', '']
            # "asdc".split("%") -> ["asdc']
            for i in range(len(text_split)):
                if i%2==0:
                    # even position is always the char that should be kept
                    new_str.append(text_split[i])
                else:
                    # odd position is always the char that is the key
                    new_str.append(graph.get(text_split[i]))
            text = "".join(new_str)
        return text

<b> 7223 Candy Crush </b>

interesting problem, find >=3 consecutive positions with same value, they are crushable, top item will drop to the crushed position. this process repeat until no crushable position found.

the solution is brute force idea, loop over each position to find positions that has same neighbords, check row level then check col level.  
mark the crushable position with value 0  
then drop the top values to the bottom by placing non-zero values starting from bottom position. Then fill rest of the position with 0

time complexity:
- each while iteration checking every cell of the board, which takes O(n*m)
-   crushing cell loops over the crush set, at most it is n*m, so it takes O(K), K at worse case is n*m
-   dropping candidates loop over each column and scan from bottom to top, in total it also loop over each position of the board, so O(n*m)
-   the while iteration takes T times, where in worse case the T can be N*M
-   in total O((N*M)*T) = O(N^2 * M^2)

space complexity:
-   crush set takes O(N*M) at most

In [4]:
from typing import List
def candyCrush(board: List[List[int]]) -> List[List[int]]:
    n, m = len(board), len(board[0])
    while True:
        # check any candies are in pattern for crushing
        crush = set()
        for r in range(n):
            for c in range(m):
                if c > 1 and board[r][c] and board[r][c] == board[r][c-1] and board[r][c]==board[r][c-2]:
                    crush |= {(r,c), (r,c-1), (r,c-2)}
                if r > 1 and board[r][c] and board[r][c] == board[r-1][c] and board[r][c] == board[r-2][c]:
                    crush |= {(r,c), (r-1, c), (r-2, c)}
        # crush by marking position to 0
        if not crush: break
        for r, c in crush: board[r][c] = 0

        # drop non-zero items downwards
        # check one column at a time
        for c in range(m):
            # starting position, all non-zero items should be at bottom
            ind = n - 1 
            for r in range(n-1, -1, -1):
                if board[r][c]:
                    board[ind][c] = board[r][c]
                    ind -= 1
            # for rest of the position, make it zero, all its original values are not already in bottom position
            for r in range(ind+1):
                board[r][c] = 0
    return board


In [5]:
board = [[1,3,5,5,2],[3,4,3,3,1],[3,2,4,5,2],[2,4,4,5,5],[1,4,4,1,1]]
candyCrush(board)

[[1, 3, 0, 0, 0],
 [3, 4, 0, 5, 2],
 [3, 2, 0, 3, 1],
 [2, 4, 0, 5, 2],
 [1, 4, 3, 1, 1]]

### Cycling Swapping 
it is an algorithm that place each element direclty into its correct position (in place) by following cycles of swaps, instead of doing many independent swaps.

while an eleent is not in tis correct position, swap it with the element currently occupying its correct position. so its swap is valid to make one position correct.

<b> 765 couples holding hands </b>

couples are defined as (0, 1), (2,3), (4,5)

for any person x, its partner is x+1 if x%2==0 else x-1

time complexity
-   only loop N once, each swap is O(1), no swap is wasted, each swap fixes one incorrect couple
-   so total O(N)

space complexity
-   O(N) for having position map.

In [11]:
def minSwapsCouples(row: List[int]) -> int:
    n = len(row)

    position = {person: i for i, person in enumerate(row)}
    swaps = 0
    for i in range(0, n, 2):
        first = row[i]
        partner = first + 1 if first%2 == 0 else first - 1
        # if partner is in right seat, continue
        if row[i+1] == partner:
            continue 
        # otherwise swap
        partner_index = position[partner]
        row[i+1], row[partner_index] = row[partner_index], row[i+1]
        # update the position map
        position[row[partner_index]] = partner_index # the previous person at i+1 now is at partner_index
        position[row[i+1]] = i+1 # the partner previously at partner index now is at i+1
        swaps += 1
    return swaps


In [12]:
minSwapsCouples([0, 2, 1, 3])

1

<b> 41 first missing positive </b>

using the cycling swapping algorithms. put the positive values to its right place, then loop again to see which position does not have the right answer, the first one met is the result.

time complexity:
-   main for loop in O(N), the inner while loop swaps each number at most once into its correct position, and each swap puts at least one number to its final place, so total number of swaps is O(N)
-   the second loop is O(N) too
-   so in total it is O(N)

space complexity:
-   O(1)



In [13]:

def firstMissingPositive(nums: List[int]) -> int:
    n = len(nums)
    for i in range(n):
        # Place nums[i] in its correct position if possible
        # only for positive values
        while 1 <= nums[i] <= n and nums[nums[i]-1] != nums[i]:
            # Swap nums[i] with nums[nums[i]-1]
            correct_idx = nums[i] - 1
            nums[i], nums[correct_idx] = nums[correct_idx], nums[i]
    # After placement, the first index i where nums[i] != i+1 is the answer
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1

In [14]:
firstMissingPositive([3, 4, -1, 1])

2

<b> 317 shortest distance from all buildings </b>

You are given an m x n grid grid of values 0, 1, or 2, where:

each 0 marks an empty land that you can pass by freely,
each 1 marks a building that you cannot pass through, and
each 2 marks an obstacle that you cannot pass through.
You want to build a house on an empty land that reaches all buildings in the shortest total travel distance. You can only move up, down, left, and right.

Return the shortest travel distance for such a house. If it is not possible to build such a house according to the above rules, return -1.

The total travel distance is the sum of the distances between the houses of the friends and the meeting point.

example:  
Input: grid = [[1,0,2,0,1],[0,0,0,0,0],[0,0,1,0,0]]  
Output: 7



solution 1: BFS from each valid house, keep updating the distance and reach matrix. the is_visited matrix is reset every time a new BFS start.

time complexity:
- N is row of grid, M is col of grid, B is number of house in the grid
- for each house, it act as starting point and visit all 0 position at most once per BFS, which at most is N*M for each BFS
- so total is O(B*N*M)

space complexity
- distance and reach: O(N*M)
- queue: valid place to search neighbors, house as the starting point, all 0 positions can be added to queue, worse case O(M*N) at each loop level, the queue wont be greater than this size each time.
- so total O(N*M)

In [ ]:
from collections import deque
class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        """
        keep track of how many houses available
        BFS, each house is a valid starting point.
        for each house, do BFS search, check 4 direction, if it is another house, update the distance and the reach matrix
        this new house becomes next valid starting point for next level.
        notice that the is_visited matrix is reset everytime a new house is used as the starting point of the BFS
        only compare the distance of position that can reach all houses.

        """
        if not grid or grid[0] == 0: return 0

        direction = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        
        row, col = len(grid), len(grid[0])
        # check total distance of reaching all other houses when starting from each position
        distance = [[0]*col for _ in range(row)]
        # check how many houses can be reached from each position as starting point
        reach = [[0]*col for _ in range(row)]

        building_num = 0
        for i in range(row):
            for j in range(col):
                if grid[i][j] == 1:
                    building_num += 1
                    # each house is a valid starting position
                    q = deque([(i, j)])
                    is_visited = [[False]*col for _ in range(row)]
                    level = 1
                    while q:
                        q_size = len(q)
                        for _ in range(q_size):
                            row_i, col_j = q.popleft()
                            for d_i, d_j in direction:
                                next_row_i = row_i + d_i
                                next_col_j = col_j + d_j
                                if (next_row_i >=0 and next_row_i<row and next_col_j>=0 and next_col_j < col and grid[next_row_i][next_col_j]==0 and not is_visited[next_row_i][next_col_j]):
                                    # your neighbor house come to visit you, so add level for all other houes visit you. that is the final distance value of this house
                                    distance[next_row_i][next_col_j] += level
                                    # keep track of how many other houses visit you.
                                    reach[next_row_i][next_col_j] += 1
                                    is_visited[next_row_i][next_col_j]=True
                                    q.append((next_row_i, next_col_j))
                        level += 1

        shortest = math.inf
        for i in range(row):
            for j in range(col):
                if grid[i][j] == 0 and reach[i][j] == building_num:
                    shortest = min(shortest, distance[i][j])
        return shortest if shortest!=math.inf else -1

solution 2: optimized version

using a walk var to check if the position can be visited by all houses. previously we use the reach matrix to track this, now we use a single integer. after all the BFS visit from all valid house, the walk should be -(n-1) the number of houses. So for next BFS only starting with position that was visited by other BFS.

this method greatly reduce the steps because: a cell is visited only if all previous buildings could reach it

why shortest is updated within the BFS
- Only the last BFS iteration produces the final valid candidates. 
- all previous shortest calculation is invalid because the looping is not finished yet.

In [ ]:
from collections import deque
from typing import List

def shortestDistance(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    distance = [[0] * n for _ in range(m)]
    walk = 0
    directions = [(0,1),(0,-1),(1,0),(-1,0)]
    
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                shortest = float('inf')
                q = deque()
                q.append((i, j, 0))  # (x, y, dist)                    
                while q:
                    x, y, d = q.popleft()                        
                    for dx, dy in directions:
                        nx, ny = x + dx, y + dy                           
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == walk:
                            # mark visited for this BFS round
                            grid[nx][ny] -= 1
                            distance[nx][ny] += d + 1 
                            q.append((nx, ny, d + 1))                                
                            shortest = min(shortest, distance[nx][ny])                    
                walk -= 1
        
    return shortest if shortest != float('inf') else -1